In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = '/home/mkeoliya/projects/CAREBench/data/ehrshot'

BREAKDOWN EHRSHOT CSV

In [3]:
df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot.csv'), index_col=0)

/tmp/ipykernel_1824918/1023219997.py:1: DtypeWarning: Columns (3,5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot.csv'), index_col=0)


In [4]:
df = df.rename(columns={'patient_id': 'CSN', 'start': 'Time', 'code': 'Measure', 'value': 'Value', 'end': 'Result_time'})
df = df.drop(columns=['unit'])
df = df.sort_values(by=['CSN', 'Time']).reset_index(drop=True)

In [5]:
demo_df = df[df['omop_table'] == 'person']
df = df[~(df['omop_table'] == 'person')]

In [8]:
demo_df = demo_df[['CSN', 'Time', 'Measure']]

In [9]:
demo_df.to_csv(os.path.join(DATA_DIR, 'demo.csv'), index=False)
df.to_csv(os.path.join(DATA_DIR, 'ehrshot.csv'), index=False)

DEMOGRAPHICS

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot.csv'))
demo_df = pd.read_csv(os.path.join(DATA_DIR, 'demo.csv'))

In [18]:
label_df = df[['CSN']].drop_duplicates().reset_index(drop=True)

In [19]:
start_time = df[['CSN', 'Time']].groupby('CSN').min().reset_index()
label_df = label_df.merge(start_time, on='CSN')

In [21]:
birth_df = demo_df[demo_df['Measure'] == 'SNOMED/3950001'][['CSN', 'Time']]
birth_df = birth_df.rename(columns={'Time': 'Birth'})
label_df = label_df.merge(birth_df, on='CSN', how='left')

In [23]:
label_df['Time'] = pd.to_datetime(label_df['Time'])
label_df['Birth'] = pd.to_datetime(label_df['Birth'])
label_df['Age'] = label_df.apply(lambda x: x['Time'].year - x['Birth'].year - ((x['Time'].month, x['Time'].day) < (x['Birth'].month, x['Birth'].day)), axis=1)

In [24]:
gender_df = demo_df[demo_df['Measure'].str.startswith('Gender/')][['CSN', 'Measure']]
gender_df = gender_df.rename(columns={'Measure': 'Gender_str'})
label_df = label_df.merge(gender_df, on='CSN', how='left')

In [25]:
race_df = demo_df[demo_df['Measure'].str.startswith('Race/')][['CSN', 'Measure']]
race_df = race_df.rename(columns={'Measure': 'Race_str'})
label_df = label_df.merge(race_df, on='CSN', how='left')
label_df['Race_str'] = label_df['Race_str'].fillna('Race/0')

In [26]:
ethnicity_df = demo_df[demo_df['Measure'].str.startswith('Ethnicity/')][['CSN', 'Measure']]
ethnicity_df = ethnicity_df.rename(columns={'Measure': 'Ethnicity_str'})
label_df = label_df.merge(ethnicity_df, on='CSN', how='left')
label_df['Ethnicity_str'] = label_df['Ethnicity_str'].fillna('Ethnicity/None')

In [27]:
AGES = [0, 20, 30, 40, 50, 60, 70, 80, 90]
AGE_LABELS = [f'Age/{AGES[i]}-{AGES[i+1]}' for i in range(0, len(AGES)-1)]
def age_str(x):
    ind = np.searchsorted(AGES, x, 'right')
    return AGE_LABELS[ind-1]

In [28]:
label_df['Age_str'] = label_df['Age'].apply(age_str)

In [29]:
label_df['Race'] = label_df['Race_str'].apply(lambda x: x.split('/', 1)[1])
label_df['Gender'] = label_df['Gender_str'].apply(lambda x: x.split('/', 1)[1])
label_df['Ethnicity'] = label_df['Ethnicity_str'].apply(lambda x: x.split('/', 1)[1])

In [30]:
label_df['Gender_ind'] = pd.factorize(label_df['Gender'])[0]
label_df['Ethnicity_ind'] = pd.factorize(label_df['Ethnicity'])[0]
label_df['Race_ind'] = pd.factorize(label_df['Race'])[0]
label_df['Age_ind'] = label_df['Age']

In [32]:
label_df.to_csv(os.path.join(DATA_DIR, 'demo.csv'), index=False)

BUCKETS

In [3]:
import pickle

In [4]:
df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot.csv'))

In [16]:
def try_parse_float(x):
    if isinstance(x, str):
        try:
            return float(x)
        except ValueError:
            return x
    return x

In [6]:
df['Value'] = df['Value'].apply(try_parse_float)

In [7]:
str_df = df[(df['Value'].apply(type) != float)]
df = df[(df['Value'].apply(type) == float)]

In [49]:
codes = sorted(df['Measure'].unique().tolist())
scale = 10_000 # for 4 decimal places
codes_dict = {}
file_path = os.path.join(DATA_DIR, 'buckets.pkl')

In [ ]:
# This may take few hours to run
for i, c in enumerate(codes):
    df_c = df[df['Measure'] == c]
    df_c = df_c.dropna(subset=['Value'])
    if df_c['Value'].nunique() <= 1:
        codes_dict[c] = c
    else:
        bucket = np.percentile(df_c['Value'], np.arange(0, 110, 10), method='lower')
        bucket = bucket[np.insert(np.diff(bucket) != 0, 0, True)]
        bucket = np.array([
                np.floor(bucket[0] * scale) / scale,             # floor first
                *[round(b, ndigits=4) for b in bucket[1:-1]],    # round middle
                np.ceil(bucket[-1] * scale) / scale              # ceil last
            ])
        codes_dict[c] = bucket[np.concatenate(([True], bucket[1:] != bucket[:-1]))]
    
    if i % 10 == 0:
        print(f'Processing code {i+1}/{len(codes)}')

        if os.path.exists(file_path): 
            with open(file_path, 'rb') as f:
                buckets = pickle.load(f)
        else: buckets = {}
        buckets.update(codes_dict)

        with open(file_path, 'wb') as f:
            pickle.dump(buckets, f)
        codes_dict = {}

In [50]:
IGNORE_VALS = ['see text', 'comment', 'see scanned results', 'unable to interpret', 'not applicable', 'see below', 'not given', 
               'see note', 'see comments', 'final report', '(note)', 'see comment', 'not done', 'comment:', 'random', 'note', 'invalid',
               'detailed information on file in his.', 'indeterminate', 'scanned:  shc/epic media manager', 'equivocal', 'tnp']

In [51]:
def convert_value(x):
    if x == 'non reactive' or x == 'non-reactive' or x == 'nonreactive':
        return 'non-reactive'
    elif x == 'not detected' or x == 'none detected' or x == 'none' or x == 'none seen' or x == 'absent' or x == 'neg' or x == '-' or x == 'n' or x =='not detected' or x == 'clear':
        return 'negative'
    elif x == 'occ.':
        return 'occasional'
    elif x == 'dnr':
        return 'normal'
    elif x == 'intermed.':
        return 'intermediate'
    elif x == 'pos' or x == '+' or x =='present' or x == 'detected':
        return 'positive'
    else:
        return x

In [52]:
str_df['Value'] = str_df['Value'].apply(lambda x: x.lower() if not x is None else x)
str_df = str_df[~(str_df['Value'].isin(IGNORE_VALS))]
str_df['Value'] = str_df['Value'].apply(convert_value)

In [54]:
str_dict = {}
for i, code in enumerate(str_df['Measure'].unique()):
    df_c = str_df[str_df['Measure']==code]
    if len(df_c['Value'].unique()) <= 1:
        str_dict[code] = code
    else:
        str_dict[code] = df_c['Value'].unique()

In [55]:
with open(os.path.join(DATA_DIR, 'str_buckets.pkl'), 'wb') as f:
    pickle.dump(str_dict, f)

EVENTVAL

In [23]:
def bucket_eventval(event, val, d):
    buckets = d[event]
    if len(buckets) <= 1 or type(buckets) == str: 
        return event
    ind = np.searchsorted(buckets, val, side='right')
    if ind == len(buckets):
        eventval = f"{event}|{buckets[ind-1]}-"
    else:
        eventval = f"{event}|{buckets[ind-1]}-{buckets[ind]}"
    return eventval

In [24]:
def bucket_ind(event, val, d):
    buckets = d[event]
    if len(buckets) <= 1 or type(buckets) == str:
        return 0
    ind = np.searchsorted(buckets, val, side='right')
    return ind

In [43]:
def str_bucket_ind(event, val, d):
    bucket = d[event]
    if type(bucket) == str or len(bucket) == 1: 
        return 0
    try:
        ind = list(bucket).index(val)
    except:
        print((bucket))
        print(bucket)
        print(val)
    return ind

In [57]:
with open(os.path.join(DATA_DIR, 'buckets.pkl'), 'rb') as file:
    buckets = pickle.load(file)

with open(os.path.join(DATA_DIR, 'str_buckets.pkl'), 'rb') as file:
    str_buckets = pickle.load(file)

In [15]:
df = pd.read_csv(os.path.join(DATA_DIR, f'ehrshot.csv'))

In [58]:
df['Value'] = df['Value'].apply(try_parse_float)
float_df = df[df['Value'].apply(type) == float]
str_df = df[(df['Value'].apply(type) == str)]
none_df = df[df['Value'].isna()]

In [59]:
float_df.loc[:, 'eventval'] = float_df.apply(lambda x: bucket_eventval(x['Measure'], x['Value'], buckets), axis=1)
float_df.loc[:, 'bucket'] = float_df.apply(lambda x: bucket_ind(x['Measure'], x['Value'], buckets), axis=1)

/tmp/ipykernel_2358122/3373929487.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  float_df.loc[:, 'eventval'] = float_df.apply(lambda x: bucket_eventval(x['Measure'], x['Value'], buckets), axis=1)
/tmp/ipykernel_2358122/3373929487.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  float_df.loc[:, 'bucket'] = float_df.apply(lambda x: bucket_ind(x['Measure'], x['Value'], buckets), axis=1)


In [60]:
none_code = none_df[none_df['Measure'].apply(lambda x: type(buckets[x]) == str)]
none_df = none_df[~(none_df['Measure'].isin(none_code['Measure']))]
none_df = none_df[none_df['Measure'].isin(str_buckets.keys())]
none_str = none_df[none_df['Measure'].apply(lambda x: type(str_buckets[x]) == str)]
none_df = pd.concat([none_code, none_str], ignore_index=True)
none_df['eventval'] = none_df['Measure']
none_df['bucket'] = 0

In [ ]:
str_df.loc[:, 'Value'] = str_df['Value'].apply(lambda x: x.lower() if not x is None else x)
str_df = str_df[~(str_df['Value'].isin(IGNORE_VALS))]
str_df.loc[:, 'Value'] = str_df['Value'].apply(convert_value)
str_df['eventval'] = (str_df['Measure'] + '|' + str_df['Value'])
str_df['bucket'] = str_df.apply(lambda x: str_bucket_ind(x['Measure'], x['Value'], str_buckets), axis=1)

In [63]:
df = pd.concat([float_df, str_df, none_df], ignore_index=True)

In [66]:
df.to_csv(os.path.join(DATA_DIR, f'ehrshot.csv'), index=False)

Time Arrive

In [7]:
df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot.csv'))

/tmp/ipykernel_2763904/3116770236.py:1: DtypeWarning: Columns (2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(DATA_DIR, 'ehrshot.csv'))


In [ ]:
pat_df = df[df['omop_table'].isin(['visit_occurrence','visit_detail', 'death'])]

In [44]:
visits_df = pat_df[pat_df['omop_table'].isin(['visit_occurrence','visit_detail'])]
visit_duration = visits_df.groupby(['CSN', 'visit_id'])['Time'].min().reset_index()
visit_duration = visit_duration.merge(visits_df.groupby(['CSN', 'visit_id'])['Result_time'].max().reset_index(), on=['CSN', 'visit_id'])
visit_duration = visit_duration.rename(columns={'Time': 'visit_start', 'Result_time': 'visit_end'})

In [ ]:
pat_df = pat_df.drop(columns=['Value', 'eventval', 'bucket'])
pat_df = pat_df.merge(visit_duration, on=['CSN', 'visit_id'])

In [54]:
pat_df.to_csv(os.path.join(DATA_DIR, 'visits.csv'), index=False)

In [ ]:
import numpy as np

class UnionFind:
    def __init__(self, n):
        self.parent = np.arange(n)
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        pa, pb = self.find(a), self.find(b)
        if pa != pb:
            self.parent[pb] = pa

def assign_groups_fast(sub_df):
    print(len(sub_df))
    n = len(sub_df)
    uf = UnionFind(n)
    idx_map = dict(zip(sub_df.index, range(n)))
    
    # 1) Link rows with same visit_id
    for _, g in sub_df.groupby('visit_id').groups.items():
        g_idx = list(g)
        base = idx_map[g_idx[0]]
        for gi in g_idx[1:]:
            uf.union(base, idx_map[gi])
    
    # 2) Link rows within 1 day
    sorted_idx = sub_df.sort_values('Time').index.to_numpy()
    times = sub_df.loc[sorted_idx, 'Time'].to_numpy()
    for i in range(n):
        j = i + 1
        while j < n and (times[j] - times[i]) < np.timedelta64(1, 'D'):
            uf.union(idx_map[sorted_idx[i]], idx_map[sorted_idx[j]])
            j += 1
    
    # Return group numbers
    groups = [uf.find(idx_map[i]) for i in sub_df.index]
    # Normalize group IDs to start from 1 per CSN
    mapping = {root: k+1 for k, root in enumerate(np.unique(groups))}
    return pd.Series([mapping[g] for g in groups], index=sub_df.index)

In [ ]:
df['Time'] = pd.to_datetime(df['Time'])
df['group'] = df.groupby('CSN', group_keys=False).apply(assign_groups_fast)
df = df.drop_duplicates()

In [ ]:
min_time = df[['CSN', 'group', 'Time']].groupby(['CSN', 'group'], as_index=False).agg(min_time=('Time', 'min'))
df = df.merge(min_time, on=['CSN', 'group'])

In [12]:
df['Time'] = pd.to_datetime(df['Time'])
df['min_time'] = pd.to_datetime(df['min_time'])
df['time_arrive'] = (df['Time'] - df['min_time']).dt.total_seconds() / 3600

In [11]:
df = pd.read_csv(os.path.join(DATA_DIR, 'data.csv'))

/tmp/ipykernel_1354666/729247320.py:1: DtypeWarning: Columns (2,4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(DATA_DIR, 'data.csv'))


In [14]:
med_df = df[df['omop_table'].isin(['device_exposure', 'condition_occurrence', 'procedure_occurrence', 'note', 'drug_exposure'])]
num_df = df[df['omop_table'].isin(['measurement', 'observation'])]

In [15]:
assert(len(df) == len(med_df) + len(num_df))

In [16]:
med_df.to_csv(os.path.join(DATA_DIR, 'med.csv'), index=False)
num_df.to_csv(os.path.join(DATA_DIR, 'numerics.csv'), index=False)

DATA COUNT

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
N = 5

In [ ]:
counts = df.groupby('patient_id')['code'].count().reset_index().sort_values('Count')
counts = counts[counts['Count'] > N]

In [ ]:
counts['values'] = (counts['Count']/10).round().astype(int)
(counts['values'] < 40).sum() / len(counts)

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(counts['values'], bins=range(counts['values'].min(), counts['values'].max() + 2), align='left')
plt.xlabel('Events')
plt.ylabel('Frequency (Total: 116474)')
plt.grid(True, linestyle='--')
plt.show()